# BERTilda anual — comparação dos dois corpus

Piloto com janelas anuais de 2015–2017 e uma intervenção por documento. Esta implementação é uma aproximação exploratória ao BERTilda: as regras de eventos não reproduzem integralmente o artigo.


## Experiência

Executa BERTopic e o acompanhamento temporal uma vez por versão, com os mesmos IDs e parâmetros. Os embeddings já existentes são verificados por assinatura e carregados por blocos; não são gerados novamente. Os dois textos e embeddings diferem, pelo que a comparação conjunta não isola lematização, stopwords ou termos procedimentais.


In [ ]:
from pathlib import Path
import hashlib
import json
import numpy as np
import pandas as pd
from bertopic import BERTopic
from hdbscan import HDBSCAN
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import adjusted_rand_score
from umap import UMAP
from IPython.display import display


## 1. Configuração


In [ ]:
RANDOM_STATE = 42
YEARS = [2015, 2016, 2017]
VERSOES = ["contextual", "lexical"]
MODEL_NAME = "neuralmind/bert-base-portuguese-cased"
MAX_SEQ_LENGTH = 512
MIN_CLUSTER_SIZE = 50
MIN_DOCS_PER_TOPIC = 50
TOP_N_WORDS = 10
SIMILARITY_THRESHOLD = 0.70
DOCUMENT_TOPIC_THRESHOLD = 0.40
COVERAGE_THRESHOLD = 0.30
SPLIT_COVERAGE_THRESHOLD = 0.70
MERGE_COVERAGE_THRESHOLD = 0.70
VALIDATION_SAMPLE_SIZE = 30
NOME_EXPERIENCIA = "comparacao_2015_2017_portugues_v2"

UMAP_NEIGHBORS = 15
HDBSCAN_SELECTION = "eom"


## 2. Carregar corpus e embeddings

Os manifestos garantem que os embeddings correspondem ao texto e à configuração. MPS não é necessário nesta fase: os embeddings já foram criados.


In [ ]:
data_dir = next((p.resolve() for p in [Path("../../data"), Path("data")]
                 if (p / "parlamint_pt_corpus_contextual.csv").exists()), None)
if data_dir is None:
    raise FileNotFoundError("Não foram encontrados os corpus em data.")
nome_anos = "anos_" + "_".join(map(str, YEARS))
embedding_base = data_dir / "embeddings_bertimbau" / nome_anos
output_base = data_dir / "bertilda_annual" / NOME_EXPERIENCIA
if len(set(YEARS)) != len(YEARS) or YEARS != sorted(YEARS) or any(b - a != 1 for a, b in zip(YEARS, YEARS[1:])):
    raise ValueError("YEARS deve conter anos únicos, consecutivos e por ordem crescente.")
if any((output_base / versao).exists() for versao in VERSOES):
    raise FileExistsError(f"Já existem resultados em {output_base}. Escolhe outro NOME_EXPERIENCIA para os preservar.")

def assinatura_corpus(ids, textos, versao, chunk_size):
    h = hashlib.sha256()
    config = {"modelo": MODEL_NAME, "max_seq_length": MAX_SEQ_LENGTH, "pooling": "mean",
              "normalize_embeddings": True, "versao": versao, "chunk_size": chunk_size}
    h.update(json.dumps(config, sort_keys=True).encode())
    for identificador, texto in zip(ids, textos):
        for valor in (identificador, texto):
            dados = valor.encode("utf-8")
            h.update(len(dados).to_bytes(8, "big"))
            h.update(dados)
    return h.hexdigest()

def carregar_versao(versao):
    ficheiro = data_dir / f"parlamint_pt_corpus_{versao}.csv"
    corpus = pd.read_csv(ficheiro, dtype="string", keep_default_na=False)
    if not {"ID", "Date", "texto_modelo"}.issubset(corpus.columns):
        raise ValueError(f"Colunas necessárias em falta: {ficheiro}")
    corpus["ano"] = pd.to_datetime(corpus["Date"], errors="coerce").dt.year
    corpus = corpus.loc[corpus["ano"].isin(YEARS)].sort_values("ID").reset_index(drop=True)
    if corpus.empty or corpus["ID"].eq("").any() or corpus["ID"].duplicated().any() or corpus["texto_modelo"].str.strip().eq("").any():
        raise ValueError(f"Corpus vazio ou com IDs/textos inválidos: {ficheiro}")
    if set(corpus["ano"].unique()) != set(YEARS):
        raise ValueError(f"Faltam anos em {ficheiro}")
    pasta = embedding_base / versao
    manifesto = json.loads((pasta / "manifesto.json").read_text(encoding="utf-8"))
    chunk_size = int(manifesto["chunk_size"])
    ids = corpus["ID"].astype(str).tolist()
    textos = corpus["texto_modelo"].astype(str).tolist()
    assinatura = assinatura_corpus(ids, textos, versao, chunk_size)
    if manifesto["assinatura"] != assinatura or manifesto["documentos"] != len(ids) or manifesto["modelo"] != MODEL_NAME or manifesto["versao"] != versao:
        raise ValueError(f"O manifesto não corresponde ao corpus ou modelo: {pasta}")
    blocos = []
    dimensao = None
    for inicio in range(0, len(ids), chunk_size):
        fim = min(inicio + chunk_size, len(ids))
        caminho = pasta / f"bloco_{inicio:08d}_{fim:08d}.npz"
        if not caminho.exists():
            raise FileNotFoundError(f"Bloco de embeddings em falta: {caminho}")
        with np.load(caminho, allow_pickle=False) as bloco:
            vetores = bloco["embeddings"]
            if not np.array_equal(bloco["document_ids"].astype(str), np.asarray(ids[inicio:fim])):
                raise ValueError(f"IDs desalinhados no bloco: {caminho}")
            if vetores.ndim != 2 or vetores.shape[0] != fim - inicio or not np.isfinite(vetores).all():
                raise ValueError(f"Embeddings inválidos: {caminho}")
            if dimensao is None:
                dimensao = vetores.shape[1]
            if vetores.shape[1] != dimensao:
                raise ValueError(f"Dimensão inconsistente: {caminho}")
            blocos.append(vetores)
    corpus = corpus.rename(columns={"texto_modelo": "documento_modelo"})
    return corpus, np.concatenate(blocos, axis=0), assinatura

corpora = {}
for versao in VERSOES:
    corpora[versao] = carregar_versao(versao)
    print(f"{versao}: {len(corpora[versao][0]):,} intervenções; embeddings {corpora[versao][1].shape}")
if not corpora["contextual"][0]["ID"].equals(corpora["lexical"][0]["ID"]):
    raise ValueError("Os corpus não têm os mesmos IDs na mesma ordem.")
if not corpora["contextual"][0]["ano"].equals(corpora["lexical"][0]["ano"]):
    raise ValueError("As datas dos mesmos IDs divergem entre corpus.")
print("IDs, anos, textos e blocos verificados.")


## 3. BERTopic e relações entre anos


In [ ]:
try:
    portuguese_stopwords = stopwords.words("portuguese")
except LookupError as error:
    raise LookupError("Instale os stopwords com nltk.download('stopwords') e volte a executar esta célula.") from error

def make_topic_model():
    umap_model = UMAP(
        n_neighbors=UMAP_NEIGHBORS,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=RANDOM_STATE,
    )
    hdbscan_model = HDBSCAN(
        min_cluster_size=MIN_CLUSTER_SIZE,
        metric="euclidean",
        prediction_data=True,
        cluster_selection_method=HDBSCAN_SELECTION,
    )
    vectorizer_model = CountVectorizer(
        stop_words=portuguese_stopwords,
        ngram_range=(1, 2),
        min_df=2,
    )
    return BERTopic(
        language="multilingual",
        embedding_model=None,
        umap_model=umap_model,
        hdbscan_model=hdbscan_model,
        vectorizer_model=vectorizer_model,
        calculate_probabilities=False,
        verbose=True,
    )

In [ ]:
def forward_coverage(window_1, window_2):
    coverage = {}
    target_ids = list(window_2["topic_embeddings"].keys())
    if not target_ids:
        return coverage

    target_centroids = np.vstack([window_2["topic_embeddings"][topic] for topic in target_ids])
    source_docs = window_1["documents_with_topics"]

    for source_topic in window_1["topic_embeddings"]:
        embeddings = np.vstack(source_docs.loc[source_docs["topic_id"] == source_topic, "embedding"].to_numpy())
        similarity = cosine_similarity(embeddings, target_centroids)
        best_target = similarity.argmax(axis=1)
        best_score = similarity.max(axis=1)
        valid = best_score >= DOCUMENT_TOPIC_THRESHOLD
        counts = np.bincount(best_target[valid], minlength=len(target_ids))
        for index, count in enumerate(counts):
            if count:
                coverage[(source_topic, target_ids[index])] = count / len(embeddings)
    return coverage

def backward_coverage(window_1, window_2):
    coverage = {}
    source_ids = list(window_1["topic_embeddings"].keys())
    if not source_ids:
        return coverage

    source_centroids = np.vstack([window_1["topic_embeddings"][topic] for topic in source_ids])
    target_docs = window_2["documents_with_topics"]

    for target_topic in window_2["topic_embeddings"]:
        embeddings = np.vstack(target_docs.loc[target_docs["topic_id"] == target_topic, "embedding"].to_numpy())
        similarity = cosine_similarity(embeddings, source_centroids)
        best_source = similarity.argmax(axis=1)
        best_score = similarity.max(axis=1)
        valid = best_score >= DOCUMENT_TOPIC_THRESHOLD
        counts = np.bincount(best_source[valid], minlength=len(source_ids))
        for index, count in enumerate(counts):
            if count:
                coverage[(source_ids[index], target_topic)] = count / len(embeddings)
    return coverage

Os modelos usam os mesmos parâmetros em cada versão. Os resultados de cada execução são guardados imediatamente numa pasta separada; um nome de experiência existente é preservado.


In [ ]:
def executar_versao(versao):
    print(f"\n=== {versao.upper()} ===")
    corpus_run, document_embeddings, assinatura = corpora[versao]
    window_topics = []
    assignments = []

    for year in YEARS:
        indices = np.flatnonzero(corpus_run["ano"].to_numpy() == year)
        subset = corpus_run.iloc[indices].copy().reset_index(drop=True)
        docs_window = subset["documento_modelo"].tolist()
        embeddings_window = document_embeddings[indices]

        topic_model = make_topic_model()
        topics, _ = topic_model.fit_transform(docs_window, embeddings=embeddings_window)

        subset["topic_id"] = topics
        subset["embedding"] = list(embeddings_window)
        assignments.append(subset)

        topic_info = topic_model.get_topic_info()
        valid_topic_ids = topic_info.loc[
            (topic_info["Topic"] != -1) & (topic_info["Count"] >= MIN_DOCS_PER_TOPIC),
            "Topic",
        ].tolist()

        topic_embeddings = {
            topic_id: embeddings_window[np.array(topics) == topic_id].mean(axis=0)
            for topic_id in valid_topic_ids
        }
        topic_words = {
            topic_id: [word for word, _ in topic_model.get_topic(topic_id)[:TOP_N_WORDS]]
            for topic_id in valid_topic_ids
        }
        topic_doc_counts = {
            int(row.Topic): int(row.Count)
            for row in topic_info.itertuples()
            if row.Topic in valid_topic_ids
        }

        window_topics.append({
            "year": int(year),
            "num_docs": len(subset),
            "num_noise_docs": int((subset["topic_id"] == -1).sum()),
            "topic_words": topic_words,
            "topic_doc_counts": topic_doc_counts,
            "topic_embeddings": topic_embeddings,
            "documents_with_topics": subset,
        })

    topic_assignments = pd.concat(assignments, ignore_index=True)
    print(f"Janelas modeladas: {len(window_topics)}")

    diagnostics = pd.DataFrame([
        {
            "ano": window["year"],
            "documentos": window["num_docs"],
            "topicos_validos": len(window["topic_embeddings"]),
            "documentos_ruido": window["num_noise_docs"],
            "proporcao_ruido": window["num_noise_docs"] / window["num_docs"],
        }
        for window in window_topics
    ]).sort_values("ano")

    display(diagnostics)

    if (diagnostics["topicos_validos"] == 0).any():
        print("Aviso: há anos sem tópicos válidos. Reveja MIN_CLUSTER_SIZE e MIN_DOCS_PER_TOPIC.")
    if (diagnostics["proporcao_ruido"] > 0.50).any():
        print("Aviso: há anos com mais de 50% de documentos classificados como ruído.")

    edges = []
    events = []

    for window_1, window_2 in zip(window_topics[:-1], window_topics[1:]):
        source_ids = list(window_1["topic_embeddings"].keys())
        target_ids = list(window_2["topic_embeddings"].keys())
        if not source_ids or not target_ids:
            continue

        source_centroids = np.vstack([window_1["topic_embeddings"][topic] for topic in source_ids])
        target_centroids = np.vstack([window_2["topic_embeddings"][topic] for topic in target_ids])
        similarities = cosine_similarity(source_centroids, target_centroids)
        coverage_forward = forward_coverage(window_1, window_2)
        coverage_backward = backward_coverage(window_1, window_2)

        for source_index, source_topic in enumerate(source_ids):
            for target_index, target_topic in enumerate(target_ids):
                similarity = similarities[source_index, target_index]
                forward = coverage_forward.get((source_topic, target_topic), 0)
                backward = coverage_backward.get((source_topic, target_topic), 0)
                if (
                    similarity >= SIMILARITY_THRESHOLD
                    and forward >= COVERAGE_THRESHOLD
                    and backward >= COVERAGE_THRESHOLD
                ):
                    edges.append({
                        "ano_origem": window_1["year"],
                        "topico_origem": source_topic,
                        "palavras_origem": ", ".join(window_1["topic_words"][source_topic]),
                        "ano_destino": window_2["year"],
                        "topico_destino": target_topic,
                        "palavras_destino": ", ".join(window_2["topic_words"][target_topic]),
                        "similaridade": similarity,
                        "cobertura_forward": forward,
                        "cobertura_backward": backward,
                    })

    edges_df = pd.DataFrame(edges)

    for window_1, window_2 in zip(window_topics[:-1], window_topics[1:]):
        links = edges_df[
            (edges_df["ano_origem"] == window_1["year"])
            & (edges_df["ano_destino"] == window_2["year"])
        ] if not edges_df.empty else pd.DataFrame()

        for source_topic in window_1["topic_embeddings"]:
            successors = links[links["topico_origem"] == source_topic] if not links.empty else links
            if successors.empty:
                relation = "desaparecimento"
            elif len(successors) == 1:
                relation = "continuação"
            elif successors["cobertura_forward"].sum() >= SPLIT_COVERAGE_THRESHOLD:
                relation = "divisão"
            else:
                relation = "ambíguo"

            events.append({
                "tipo": relation,
                "ano_origem": window_1["year"],
                "topico_origem": source_topic,
                "palavras_origem": ", ".join(window_1["topic_words"][source_topic]),
                "ano_destino": window_2["year"],
                "topicos_destino": successors["topico_destino"].tolist() if not successors.empty else [],
                "palavras_destino": successors["palavras_destino"].tolist() if not successors.empty else [],
                "cobertura_forward_total": successors["cobertura_forward"].sum() if not successors.empty else 0,
                "cobertura_backward_total": successors["cobertura_backward"].sum() if not successors.empty else 0,
            })

        for target_topic in window_2["topic_embeddings"]:
            predecessors = links[links["topico_destino"] == target_topic] if not links.empty else links
            if (
                len(predecessors) >= 2
                and predecessors["cobertura_backward"].sum() >= MERGE_COVERAGE_THRESHOLD
            ):
                events.append({
                    "tipo": "fusão",
                    "ano_origem": window_1["year"],
                    "topico_origem": predecessors["topico_origem"].tolist(),
                    "palavras_origem": predecessors["palavras_origem"].tolist(),
                    "ano_destino": window_2["year"],
                    "topicos_destino": [target_topic],
                    "palavras_destino": [", ".join(window_2["topic_words"][target_topic])],
                    "cobertura_forward_total": predecessors["cobertura_forward"].sum(),
                    "cobertura_backward_total": predecessors["cobertura_backward"].sum(),
                })

    events_df = pd.DataFrame(events)
    if not events_df.empty:
        display(events_df["tipo"].value_counts().rename("eventos").to_frame())

    if events_df.empty:
        validation_sample = events_df.copy()
    else:
        validation_sample = events_df.sample(
            n=min(VALIDATION_SAMPLE_SIZE, len(events_df)),
            random_state=RANDOM_STATE,
        ).sort_values(["ano_origem", "tipo"])

    display(validation_sample)

    topicos = pd.DataFrame([
        {"ano": w["year"], "topico": t, "intervencoes": w["topic_doc_counts"][t],
         "palavras": ", ".join(w["topic_words"][t])}
        for w in window_topics for t in w["topic_words"]
    ], columns=["ano", "topico", "intervencoes", "palavras"])
    output_dir = output_base / versao
    output_dir.mkdir(parents=True, exist_ok=False)
    diagnostics.to_csv(output_dir / "diagnosticos.csv", index=False)
    topic_assignments.drop(columns="embedding").to_csv(output_dir / "atribuicoes.csv", index=False)
    topicos.to_csv(output_dir / "topicos.csv", index=False)
    edges_df.to_csv(output_dir / "ligacoes.csv", index=False)
    events_df.to_csv(output_dir / "eventos.csv", index=False)
    validation_sample.to_csv(output_dir / "amostra_relacoes.csv", index=False)
    print(f"Resultados guardados: {output_dir}")
    return diagnostics, topic_assignments.drop(columns="embedding"), topicos, events_df, assinatura


In [ ]:
resultados = {}
for versao in VERSOES:
    resultados[versao] = executar_versao(versao)


## 4. Comparação

Comparam-se o número de tópicos, ruído, palavras e eventos. O índice ARI avalia a concordância das atribuições aos grupos entre versões, apenas nas intervenções que não são ruído em nenhuma delas; valores diferentes não demonstram por si só qual modelo é melhor. Os números de tópico são arbitrários. As palavras e os textos dos grupos e relações devem ser inspecionados manualmente.


In [ ]:
comparacao = pd.concat([
    diagnosticos.assign(versao=versao)
    for versao, (diagnosticos, atribuicoes, topicos, eventos, assinatura) in resultados.items()
], ignore_index=True).sort_values(["ano", "versao"])
display(comparacao)

pares = resultados["contextual"][1][["ID", "ano", "topic_id"]].merge(
    resultados["lexical"][1][["ID", "ano", "topic_id"]], on=["ID", "ano"],
    suffixes=("_contextual", "_lexical"), validate="one_to_one")
if len(pares) != len(resultados["contextual"][1]) or len(pares) != len(resultados["lexical"][1]):
    raise ValueError("As atribuições das duas versões não cobrem os mesmos IDs.")
acordo = []
for ano, dados in pares.groupby("ano"):
    ambos = dados.loc[(dados["topic_id_contextual"] != -1) & (dados["topic_id_lexical"] != -1)]
    acordo.append({"ano": ano, "intervencoes_em_ambos_topicos": len(ambos),
                   "proporcao_em_ambos_topicos": len(ambos) / len(dados),
                   "ari_sem_ruido": adjusted_rand_score(ambos["topic_id_contextual"], ambos["topic_id_lexical"]) if len(ambos) >= 2 else np.nan})
acordo = pd.DataFrame(acordo)
display(acordo)
comparacao.to_csv(output_base / "comparacao_diagnosticos.csv", index=False)
acordo.to_csv(output_base / "comparacao_agrupamentos.csv", index=False)
for versao, (_, _, topicos, eventos, _) in resultados.items():
    print(f"\nTópicos mais frequentes: {versao}")
    display(topicos.sort_values(["ano", "intervencoes"], ascending=[True, False]).groupby("ano").head(5))
    print(f"Eventos: {versao}")
    display(eventos["tipo"].value_counts().to_frame("eventos") if not eventos.empty else "Sem eventos")


## Interpretação

Examina os ficheiros `topicos.csv`, `atribuicoes.csv` e `amostra_relacoes.csv` de cada versão. Lê intervenções dos tópicos e das relações antes de concluir sobre qualidade ou drift. Ruído baixo e mais tópicos não significam automaticamente melhor resultado.
